# 循环神经网络：逐步构建 🔄

## 学习目标 🎯

完成本实验后，你将能够：

- 描述循环神经网络（RNN）的基本结构与工作原理
- **实现** RNN 单元及完整 RNN 的前向传播
- 理解长短期记忆网络（LSTM）中各门控机制的作用
- **实现** LSTM 单元及完整 LSTM 的前向传播
- （可选）推导并实现 RNN/LSTM 的反向传播

> 💡 本实验以 NumPy 手工实现所有模块，有助于深入理解序列模型的内部机制。


## 目录

- [0 - 导入包](#0)
- [1 - 基础循环神经网络的前向传播](#1)
    - [1.1 - RNN 单元](#1-1)
        - [练习 1 - rnn_cell_forward](#ex-1)
    - [1.2 - RNN 完整前向传播](#1-2)
        - [练习 2 - rnn_forward](#ex-2)
- [2 - 长短期记忆网络（LSTM）](#2)
    - [2.1 - LSTM 单元](#2-1)
        - [练习 3 - lstm_cell_forward](#ex-3)
    - [2.2 - LSTM 完整前向传播](#2-2)
        - [练习 4 - lstm_forward](#ex-4)
- [3 - 反向传播（进阶选做）](#3)


<a name='0'></a>
## 0 - 导入包 📦

运行下方代码加载本实验所需的全部依赖：

- **numpy**：数值计算核心库
- **rnn_utils**：本实验辅助函数（`softmax`、`sigmoid`）
- **public_tests**：自动评测模块


In [39]:
import numpy as np
from rnn_utils import *
from public_tests import *

np.random.seed(1)


<a name='1'></a>
## 1 - 基础循环神经网络的前向传播 ➡️

循环神经网络通过**隐藏状态**在时间步之间传递信息，使模型具备处理序列数据的能力。

基础 RNN 的整体结构，输入序列长度 $T_x$，模型在每个时间步 $t$ 重复使用同一组参数。


### 张量维度说明

| 符号 | 含义 | 形状 |
|------|------|------|
| $x^{\langle t \rangle}$ | 时间步 $t$ 的输入 | $(n_x, m)$ |
| $a^{\langle t \rangle}$ | 时间步 $t$ 的隐藏状态 | $(n_a, m)$ |
| $\hat{y}^{\langle t \rangle}$ | 时间步 $t$ 的预测输出 | $(n_y, m)$ |
| $x$ | 完整输入序列 | $(n_x, m, T_x)$ |
| $a$ | 完整隐藏状态序列 | $(n_a, m, T_x)$ |

其中 $m$ 为 mini-batch 样本数。


<a name='1-1'></a>
### 1.1 - RNN 单元

RNN 的核心是一个在每个时间步重复执行的计算单元

单步计算公式：

$$a^{\langle t \rangle} = \tanh\!\left(W_{aa}\, a^{\langle t-1 \rangle} + W_{ax}\, x^{\langle t \rangle} + b_a\right)$$

$$\hat{y}^{\langle t \rangle} = \text{softmax}\!\left(W_{ya}\, a^{\langle t \rangle} + b_y\right)$$

**参数形状**：

| 参数 | 形状 | 说明 |
|------|------|------|
| $W_{aa}$ | $(n_a, n_a)$ | 隐藏状态到隐藏状态 |
| $W_{ax}$ | $(n_a, n_x)$ | 输入到隐藏状态 |
| $W_{ya}$ | $(n_y, n_a)$ | 隐藏状态到输出 |
| $b_a$ | $(n_a, 1)$ | 隐藏层偏置 |
| $b_y$ | $(n_y, 1)$ | 输出层偏置 |


<a name='ex-1'></a>
### 练习 1 — `rnn_cell_forward`

实现单步 RNN 单元的前向传播。

**步骤**：
1. 计算新隐藏状态：$a^{\langle t \rangle} = \tanh(W_{aa}\, a^{\langle t-1 \rangle} + W_{ax}\, x^{\langle t \rangle} + b_a)$
2. 计算预测输出：$\hat{y}^{\langle t \rangle} = \text{softmax}(W_{ya}\, a^{\langle t \rangle} + b_y)$
3. 将中间量存入 cache 供反向传播使用：`cache = (a_next, a_prev, xt, parameters)`
4. 返回 `a_next, yt_pred, cache`

> 💡 提示：使用 `np.dot` 进行矩阵乘法，`np.tanh` 计算双曲正切，`softmax` 函数已从 `rnn_utils` 导入。


In [40]:
def rnn_cell_forward(xt, a_prev, parameters):
    """
    单步 RNN 单元前向传播。

    参数:
        xt         -- 当前时间步输入，形状 (n_x, m)
        a_prev     -- 上一时间步隐藏状态，形状 (n_a, m)
        parameters -- 字典，包含 Waa, Wax, Wya, ba, by

    返回:
        a_next   -- 新隐藏状态，形状 (n_a, m)
        yt_pred  -- 当前时间步预测，形状 (n_y, m)
        cache    -- (a_next, a_prev, xt, parameters)
    """
    Waa = parameters['Waa']
    Wax = parameters['Wax']
    Wya = parameters['Wya']
    ba  = parameters['ba']
    by  = parameters['by']

    ### START CODE HERE ###
    ### END CODE HERE ###

    cache = (a_next, a_prev, xt, parameters)
    return a_next, yt_pred, cache


In [41]:
# 快速功能验证
np.random.seed(1)
xt_tmp = np.random.randn(3, 10)
a_prev_tmp = np.random.randn(5, 10)
parameters_tmp = {
    'Waa': np.random.randn(5, 5),
    'Wax': np.random.randn(5, 3),
    'Wya': np.random.randn(2, 5),
    'ba':  np.random.randn(5, 1),
    'by':  np.random.randn(2, 1),
}

a_next_tmp, yt_pred_tmp, cache_tmp = rnn_cell_forward(xt_tmp, a_prev_tmp, parameters_tmp)
print("a_next[4] =\n", a_next_tmp[4])
print("a_next.shape =", a_next_tmp.shape)
print("yt_pred[1] =\n", yt_pred_tmp[1])
print("yt_pred.shape =", yt_pred_tmp.shape)


a_next[4] =
 [ 0.59584544  0.18141802  0.61311866  0.99808218  0.85016201  0.99980978
 -0.18887155  0.99815551  0.6531151   0.82872037]
a_next.shape = (5, 10)
yt_pred[1] =
 [0.9888161  0.01682021 0.21140899 0.36817467 0.98988387 0.88945212
 0.36920224 0.9966312  0.9982559  0.17746526]
yt_pred.shape = (2, 10)


In [42]:
rnn_cell_forward_tests(rnn_cell_forward)


All tests passed


**期望输出**
```
a_next[4] = 
 [ 0.59584544  0.18141802  0.61311866  0.99808218  0.85016201  0.99980978
 -0.18887155  0.99815551  0.6531151   0.82872037]
a_next.shape = (5, 10)
yt_pred[1] =
 [ 0.9888161   0.01682021  0.21140899  0.36817467  0.98988387  0.88945212
  0.36920224  0.9966312   0.9982559   0.17746526]
yt_pred.shape = (2, 10)
```


<a name='1-2'></a>
### 1.2 - RNN 完整前向传播

将 RNN 单元在 $T_x$ 个时间步上循环展开，即得到完整的 RNN 前向传播：


**算法步骤**：
1. 初始化输出张量 $a$（形状 $(n_a, m, T_x)$）和 $\hat{y}$（形状 $(n_y, m, T_x)$）为零矩阵
2. 将初始隐藏状态 $a_0$ 赋给 $a_{\text{prev}}$
3. 对每个时间步 $t = 0, 1, \ldots, T_x - 1$：
   - 取 $x^{\langle t \rangle} = x[:, :, t]$，形状 $(n_x, m)$
   - 调用 `rnn_cell_forward` 更新隐藏状态
   - 保存 $a^{\langle t \rangle}$ 和 $\hat{y}^{\langle t \rangle}$
   - 将当前 cache 追加到 caches 列表
4. 返回 `a, y_pred, caches`，其中 `caches = (列表, x)`


<a name='ex-2'></a>
### 练习 2 — `rnn_forward`

实现完整 RNN 的前向传播，在 $T_x$ 个时间步上循环调用 `rnn_cell_forward`。

**输入**：
- `x`：完整输入序列，形状 $(n_x, m, T_x)$
- `a0`：初始隐藏状态，形状 $(n_a, m)$
- `parameters`：参数字典

**输出**：
- `a`：所有时间步的隐藏状态，形状 $(n_a, m, T_x)$
- `y_pred`：所有时间步的预测，形状 $(n_y, m, T_x)$
- `caches`：`(cache_list, x)`，供反向传播使用

> 💡 提示：用 `x.shape` 获取维度，用 `np.zeros` 初始化输出张量，用列表存储每步 cache。


In [43]:
def rnn_forward(x, a0, parameters):
    """
    在 T_x 个时间步上展开 RNN，执行完整前向传播。

    参数:
        x          -- 输入序列，形状 (n_x, m, T_x)
        a0         -- 初始隐藏状态，形状 (n_a, m)
        parameters -- 参数字典

    返回:
        a      -- 所有时间步隐藏状态，形状 (n_a, m, T_x)
        y_pred -- 所有时间步预测，形状 (n_y, m, T_x)
        caches -- (cache_list, x)
    """
    caches = []
    n_x, m, T_x = x.shape
    n_y, n_a = parameters['Wya'].shape

    ### START CODE HERE ###

    ### END CODE HERE ###

    caches = (caches, x)
    return a, y_pred, caches


In [44]:
np.random.seed(1)
x_tmp = np.random.randn(3, 10, 4)
a0_tmp = np.random.randn(5, 10)
parameters_tmp = {
    'Waa': np.random.randn(5, 5),
    'Wax': np.random.randn(5, 3),
    'Wya': np.random.randn(2, 5),
    'ba':  np.random.randn(5, 1),
    'by':  np.random.randn(2, 1),
}

a_tmp, y_pred_tmp, caches_tmp = rnn_forward(x_tmp, a0_tmp, parameters_tmp)
print("a[4][1] =\n", a_tmp[4][1])
print("a.shape =", a_tmp.shape)
print("y_pred[1][3] =\n", y_pred_tmp[1][3])
print("y_pred.shape =", y_pred_tmp.shape)
print("caches[1][1][3] =\n", caches_tmp[1][1][3])
print("len(caches) =", len(caches_tmp))


a[4][1] =
 [-0.99999375  0.77911235 -0.99861469 -0.99833267]
a.shape = (5, 10, 4)
y_pred[1][3] =
 [0.79560373 0.86224861 0.11118257 0.81515947]
y_pred.shape = (2, 10, 4)
caches[1][1][3] =
 [-1.1425182  -0.34934272 -0.20889423  0.58662319]
len(caches) = 2


In [45]:
rnn_forward_test(rnn_forward)


All tests passed


**期望输出**
```
a[4][1] = 
 [-0.99999375  0.77911235 -0.99861469 -0.99833267]
a.shape = (5, 10, 4)
y_pred[1][3] =
 [ 0.79560373  0.86224861  0.11118257  0.81515947]
y_pred.shape = (2, 10, 4)
caches[1][1][3] =
 [-1.1425182  -0.34934272 -0.20889423  0.58662319]
len(caches) = 2
```


### 小结 🎉

你已经成功实现了基础 RNN 的前向传播！

基础 RNN 适用于短依赖场景。对于需要**长程依赖**的任务（如机器翻译、语音识别），它会遭遇**梯度消失**问题。下一节我们将学习能有效解决这一问题的 LSTM 网络。


<a name='2'></a>
## 2 - 长短期记忆网络（LSTM）🧠

LSTM 通过引入**细胞状态**（cell state）和三个**门控机制**来解决基础 RNN 的梯度消失问题。


### 门控机制概览

设拼接向量 $\text{concat} = \begin{bmatrix} a^{\langle t-1 \rangle} \\ x^{\langle t \rangle} \end{bmatrix}$，各门控方程如下：

**遗忘门** $\Gamma_f$（决定丢弃多少历史信息）：
$$\Gamma_f^{\langle t \rangle} = \sigma\!\left(W_f\, [a^{\langle t-1 \rangle},\, x^{\langle t \rangle}] + b_f\right)$$

**输入门** $\Gamma_i$（决定写入多少新信息）：
$$\Gamma_i^{\langle t \rangle} = \sigma\!\left(W_i\, [a^{\langle t-1 \rangle},\, x^{\langle t \rangle}] + b_i\right)$$

**候选细胞状态** $\tilde{c}^{\langle t \rangle}$：
$$\tilde{c}^{\langle t \rangle} = \tanh\!\left(W_c\, [a^{\langle t-1 \rangle},\, x^{\langle t \rangle}] + b_c\right)$$

**细胞状态更新**：
$$c^{\langle t \rangle} = \Gamma_f^{\langle t \rangle} \odot c^{\langle t-1 \rangle} + \Gamma_i^{\langle t \rangle} \odot \tilde{c}^{\langle t \rangle}$$

**输出门** $\Gamma_o$（决定输出哪部分细胞状态）：
$$\Gamma_o^{\langle t \rangle} = \sigma\!\left(W_o\, [a^{\langle t-1 \rangle},\, x^{\langle t \rangle}] + b_o\right)$$

**隐藏状态更新**：
$$a^{\langle t \rangle} = \Gamma_o^{\langle t \rangle} \odot \tanh\!\left(c^{\langle t \rangle}\right)$$

**输出预测**：
$$\hat{y}^{\langle t \rangle} = \text{softmax}\!\left(W_y\, a^{\langle t \rangle} + b_y\right)$$

其中 $\odot$ 表示逐元素乘法（Hadamard 积），$\sigma$ 为 sigmoid 函数。


<a name='2-1'></a>
### 2.1 - LSTM 单元

**LSTM 参数形状**（$n_c = n_a$）：

| 参数 | 形状 | 说明 |
|------|------|------|
| $W_f$ | $(n_a, n_a + n_x)$ | 遗忘门权重 |
| $W_i$ | $(n_a, n_a + n_x)$ | 输入门权重 |
| $W_c$ | $(n_a, n_a + n_x)$ | 候选值权重 |
| $W_o$ | $(n_a, n_a + n_x)$ | 输出门权重 |
| $W_y$ | $(n_y, n_a)$ | 输出层权重 |
| $b_f, b_i, b_c, b_o$ | $(n_a, 1)$ | 各门偏置 |
| $b_y$ | $(n_y, 1)$ | 输出偏置 |

cache 返回顺序：`(a_next, c_next, a_prev, c_prev, ft, it, cct, ot, xt, parameters)`


<a name='ex-3'></a>
### 练习 3 — `lstm_cell_forward`

实现单步 LSTM 单元的前向传播。

**步骤**：
1. 将 $a^{\langle t-1 \rangle}$ 与 $x^{\langle t \rangle}$ 沿行方向拼接：`concat = np.concatenate([a_prev, xt], axis=0)`
2. 按上述公式依次计算遗忘门 `ft`、输入门 `it`、候选值 `cct`
3. 更新细胞状态 `c_next = ft * c_prev + it * cct`
4. 计算输出门 `ot`，更新隐藏状态 `a_next = ot * np.tanh(c_next)`
5. 计算预测 `yt_pred = softmax(np.dot(Wy, a_next) + by)`
6. 按顺序打包 cache：`(a_next, c_next, a_prev, c_prev, ft, it, cct, ot, xt, parameters)`

> 💡 注意：输入门权重 `Wi`、偏置 `bi` 对应更新门（没有 `Wu`/`bu`）。


In [46]:
def lstm_cell_forward(xt, a_prev, c_prev, parameters):
    """
    单步 LSTM 单元前向传播。

    参数:
        xt         -- 当前时间步输入，形状 (n_x, m)
        a_prev     -- 上一时间步隐藏状态，形状 (n_a, m)
        c_prev     -- 上一时间步细胞状态，形状 (n_a, m)
        parameters -- 字典，包含 Wf,bf, Wi,bi, Wc,bc, Wo,bo, Wy,by

    返回:
        a_next  -- 新隐藏状态，形状 (n_a, m)
        c_next  -- 新细胞状态，形状 (n_a, m)
        yt_pred -- 当前时间步预测，形状 (n_y, m)
        cache   -- (a_next, c_next, a_prev, c_prev, ft, it, cct, ot, xt, parameters)
    """
    Wf = parameters['Wf']
    bf = parameters['bf']
    Wi = parameters['Wi']
    bi = parameters['bi']
    Wc = parameters['Wc']
    bc = parameters['bc']
    Wo = parameters['Wo']
    bo = parameters['bo']
    Wy = parameters['Wy']
    by = parameters['by']

    n_x, m = xt.shape
    n_y, n_a = Wy.shape

    ### START CODE HERE ###


    ### END CODE HERE ###

    cache = (a_next, c_next, a_prev, c_prev, ft, it, cct, ot, xt, parameters)
    return a_next, c_next, yt_pred, cache


In [47]:
np.random.seed(1)
xt_tmp = np.random.randn(3, 10)
a_prev_tmp = np.random.randn(5, 10)
c_prev_tmp = np.random.randn(5, 10)
parameters_tmp = {
    'Wf': np.random.randn(5, 5 + 3),
    'bf': np.random.randn(5, 1),
    'Wi': np.random.randn(5, 5 + 3),
    'bi': np.random.randn(5, 1),
    'Wo': np.random.randn(5, 5 + 3),
    'bo': np.random.randn(5, 1),
    'Wc': np.random.randn(5, 5 + 3),
    'bc': np.random.randn(5, 1),
    'Wy': np.random.randn(2, 5),
    'by': np.random.randn(2, 1),
}

a_next_tmp, c_next_tmp, yt_tmp, cache_tmp = lstm_cell_forward(
    xt_tmp, a_prev_tmp, c_prev_tmp, parameters_tmp
)
print("a_next[4] =\n", a_next_tmp[4])
print("a_next.shape =", a_next_tmp.shape)
print("c_next[2] =\n", c_next_tmp[2])
print("c_next.shape =", c_next_tmp.shape)
print("yt[1] =\n", yt_tmp[1])
print("yt.shape =", yt_tmp.shape)
print("cache[1][3] =\n", cache_tmp[1][3])
print("len(cache) =", len(cache_tmp))


a_next[4] =
 [-0.66408471  0.0036921   0.02088357  0.22834167 -0.85575339  0.00138482
  0.76566531  0.34631421 -0.00215674  0.43827275]
a_next.shape = (5, 10)
c_next[2] =
 [ 0.63267805  1.00570849  0.35504474  0.20690913 -1.64566718  0.11832942
  0.76449811 -0.0981561  -0.74348425 -0.26810932]
c_next.shape = (5, 10)
yt[1] =
 [0.79913913 0.15986619 0.22412122 0.15606108 0.97057211 0.31146381
 0.00943007 0.12666353 0.39380172 0.07828381]
yt.shape = (2, 10)
cache[1][3] =
 [-0.16263996  1.03729328  0.72938082 -0.54101719  0.02752074 -0.30821874
  0.07651101 -1.03752894  1.41219977 -0.37647422]
len(cache) = 10


In [48]:
lstm_cell_forward_test(lstm_cell_forward)


All tests passed


**期望输出**
```
a_next[4] = 
 [-0.66408471  0.0036921   0.02088357  0.22834167 -0.85575339  0.00138482
  0.76566531  0.34631421 -0.00215674  0.43827275]
a_next.shape = (5, 10)
c_next[2] = 
 [ 0.63267805  1.00570849  0.35504474  0.20690913 -1.64566718  0.11832942
  0.76449811 -0.0981561  -0.74348425 -0.26810932]
c_next.shape = (5, 10)
yt[1] = [ 0.79913913  0.15986619  0.22412122  0.15606108  0.97057211  0.31146381
  0.00943007  0.12666353  0.39380172  0.07828381]
yt.shape = (2, 10)
cache[1][3] =
 [-0.16263996  1.03729328  0.72938082 -0.54101719  0.02752074 -0.30821874
  0.07651101 -1.03752894  1.41219977 -0.37647422]
len(cache) = 10
```


<a name='2-2'></a>
### 2.2 - LSTM 完整前向传播

与 RNN 类似，将 LSTM 单元在 $T_x$ 个时间步上展开：


需要额外维护**细胞状态** $c$（形状 $(n_a, m, T_x)$），初始细胞状态 $c_0$ 设为全零矩阵。


<a name='ex-4'></a>
### 练习 4 — `lstm_forward`

实现完整 LSTM 的前向传播。

**步骤**：
1. 从 `x.shape` 和 `parameters` 获取维度 $n_x, m, T_x, n_y, n_a$
2. 初始化 `a`（形状 $(n_a, m, T_x)$）、`c`（形状 $(n_a, m, T_x)$）、`y`（形状 $(n_y, m, T_x)$）为零矩阵
3. 设 `a_next = a0`，`c_next = np.zeros((n_a, m))`
4. 对每个时间步 $t = 0, \ldots, T_x - 1$：
   - 取 $x^{\langle t \rangle} = x[:, :, t]$
   - 调用 `lstm_cell_forward`，获得 `a_next, c_next, yt, cache`
   - 保存 `a[:,:,t]`、`c[:,:,t]`、`y[:,:,t]`
   - 追加 cache 到列表
5. 返回 `a, y, c, caches`，其中 `caches = (cache_list, x)`

> 💡 注意返回顺序：`a, y, c, caches`（区别于 rnn_forward 的 `a, y_pred, caches`）。


In [49]:
def lstm_forward(x, a0, parameters):
    """
    在 T_x 个时间步上展开 LSTM，执行完整前向传播。

    参数:
        x          -- 输入序列，形状 (n_x, m, T_x)
        a0         -- 初始隐藏状态，形状 (n_a, m)
        parameters -- 参数字典（Wf,bf,Wi,bi,Wc,bc,Wo,bo,Wy,by）

    返回:
        a      -- 所有时间步隐藏状态，形状 (n_a, m, T_x)
        y      -- 所有时间步预测，形状 (n_y, m, T_x)
        c      -- 所有时间步细胞状态，形状 (n_a, m, T_x)
        caches -- (cache_list, x)
    """
    caches = []
    n_x, m, T_x = x.shape
    n_y, n_a    = parameters['Wy'].shape

    ### START CODE HERE ###

    ### END CODE HERE ###

    caches = (caches, x)
    return a, y, c, caches


In [50]:
np.random.seed(1)
x_tmp = np.random.randn(3, 10, 7)
a0_tmp = np.random.randn(5, 10)
parameters_tmp = {
    'Wf': np.random.randn(5, 5 + 3),
    'bf': np.random.randn(5, 1),
    'Wi': np.random.randn(5, 5 + 3),
    'bi': np.random.randn(5, 1),
    'Wo': np.random.randn(5, 5 + 3),
    'bo': np.random.randn(5, 1),
    'Wc': np.random.randn(5, 5 + 3),
    'bc': np.random.randn(5, 1),
    'Wy': np.random.randn(2, 5),
    'by': np.random.randn(2, 1),
}

a_tmp, y_tmp, c_tmp, caches_tmp = lstm_forward(x_tmp, a0_tmp, parameters_tmp)
print("a[4][3][6] =", a_tmp[4][3][6])
print("a.shape =", a_tmp.shape)
print("y[1][4][3] =", y_tmp[1][4][3])
print("y.shape =", y_tmp.shape)
print("caches[1][1][1] =\n", caches_tmp[1][1][1])
print("c[1][2][1] =", c_tmp[1][2][1])
print("len(caches) =", len(caches_tmp))


a[4][3][6] = 0.17211776753291663
a.shape = (5, 10, 7)
y[1][4][3] = 0.9508734618501101
y.shape = (2, 10, 7)
caches[1][1][1] =
 [ 0.82797464  0.23009474  0.76201118 -0.22232814 -0.20075807  0.18656139
  0.41005165]
c[1][2][1] = -0.8555449167181983
len(caches) = 2


In [51]:
lstm_forward_test(lstm_forward)


All tests passed


**期望输出**
```
a[4][3][6] = 0.172117767533
a.shape = (5, 10, 7)
y[1][4][3] = 0.95087346185
y.shape = (2, 10, 7)
caches[1][1][1] =
 [ 0.82797464  0.23009474  0.76201118 -0.22232814 -0.20075807  0.18656139
  0.41005165]
c[1][2][1] = -0.855544916718
len(caches) = 2
```


### 恭喜！🎉

你已成功实现了 RNN 和 LSTM 的前向传播！

**本实验要点回顾**：

| 模型 | 核心状态 | 优点 | 局限性 |
|------|----------|------|--------|
| 基础 RNN | 隐藏状态 $a$ | 结构简单 | 梯度消失，难以捕捉长程依赖 |
| LSTM | 隐藏状态 $a$ + 细胞状态 $c$ | 门控机制有效缓解梯度消失 | 参数量更大，计算开销更高 |

LSTM 通过三个门控（遗忘门、输入门、输出门）和独立的细胞状态，实现了对历史信息的**选择性记忆与遗忘**，是序列建模中应用最广泛的基础模块之一。


<a name='3'></a>
## 3 - 反向传播（进阶选做）⚙️

> **注意**：本节为选做内容，不计入考核。现代深度学习框架（PyTorch、TensorFlow）会自动计算反向传播，工程实践中通常无需手动实现。若对推导细节感兴趣，可继续完成本节。

---

### 3.1 - 基础 RNN 反向传播

RNN 的反向传播称为**时间反向传播**（BPTT，Backpropagation Through Time）。


单步 RNN 的梯度公式（设 $a^{\langle t \rangle} = \tanh(z)$，$z = W_{ax} x^{\langle t \rangle} + W_{aa} a^{\langle t-1 \rangle} + b_a$）：

$$\text{dtanh} = da_{\text{next}} \cdot (1 - \tanh^2(z))$$

$$dW_{ax} = \text{dtanh} \cdot x^{\langle t \rangle T}, \quad dW_{aa} = \text{dtanh} \cdot a^{\langle t-1 \rangle T}$$

$$db_a = \sum_{\text{batch}} \text{dtanh}, \quad dx^{\langle t \rangle} = W_{ax}^T \cdot \text{dtanh}$$

$$da_{\text{prev}} = W_{aa}^T \cdot \text{dtanh}$$


In [52]:
def rnn_cell_backward(da_next, cache):
    """
    单步 RNN 单元反向传播（选做）。

    参数:
        da_next -- 对 a_next 的损失梯度，形状 (n_a, m)
        cache   -- rnn_cell_forward 返回的 cache

    返回:
        gradients -- 字典，包含 dxt, da_prev, dWax, dWaa, dba
    """
    a_next, a_prev, xt, parameters = cache
    Waa = parameters['Waa']
    Wax = parameters['Wax']
    Wya = parameters['Wya']
    ba  = parameters['ba']
    by  = parameters['by']

    ### START CODE HERE ###
    ### END CODE HERE ###

    gradients = {'dxt': dxt, 'da_prev': da_prev,
                 'dWax': dWax, 'dWaa': dWaa, 'dba': dba}
    return gradients


In [53]:
def rnn_backward(da, caches):
    """
    完整 RNN 时间反向传播（选做）。

    参数:
        da     -- 所有时间步隐藏状态的梯度，形状 (n_a, m, T_x)
        caches -- rnn_forward 返回的 caches

    返回:
        gradients -- 字典，包含 dx, da0, dWax, dWaa, dba
    """
    caches_list, x = caches
    n_a, m, T_x = da.shape
    n_x, m = caches_list[0][2].shape  # xt shape

    ### START CODE HERE ###


    ### END CODE HERE ###

    gradients = {'dx': dx, 'da0': da0, 'dWax': dWax, 'dWaa': dWaa, 'dba': dba}
    return gradients


### 3.2 - LSTM 反向传播（选做）

LSTM 单步反向传播需要对各门控逐一求梯度。设：

$$da_{\text{next\_total}} = da_{\text{next}} + da_{\text{从上游传来}}$$

各门的梯度推导略（可参考 LSTM 反向传播推导文献）。

以下为框架代码，欢迎尝试完整实现。


In [54]:
def lstm_cell_backward(da_next, dc_next, cache):
    """
    单步 LSTM 单元反向传播（选做）。

    参数:
        da_next -- 对 a_next 的梯度，形状 (n_a, m)
        dc_next -- 对 c_next 的梯度，形状 (n_a, m)
        cache   -- lstm_cell_forward 返回的 cache

    返回:
        gradients -- 字典，包含 dxt, da_prev, dc_prev,
                                  dWf, dbf, dWi, dbi,
                                  dWc, dbc, dWo, dbo
    """
    a_next, c_next, a_prev, c_prev, ft, it, cct, ot, xt, parameters = cache
    n_x, m = xt.shape
    n_a, m = a_next.shape

    ### START CODE HERE ###





    ### END CODE HERE ###

    gradients = {'dxt': dxt, 'da_prev': da_prev, 'dc_prev': dc_prev,
                 'dWf': dWf, 'dbf': dbf, 'dWi': dWi, 'dbi': dbi,
                 'dWc': dWc, 'dbc': dbc, 'dWo': dWo, 'dbo': dbo}
    return gradients


In [55]:
def lstm_backward(da, caches):
    """
    完整 LSTM 时间反向传播（选做）。

    参数:
        da     -- 所有时间步隐藏状态的梯度，形状 (n_a, m, T_x)
        caches -- lstm_forward 返回的 caches

    返回:
        gradients -- 字典，包含 dx, da0, dc0 及所有权重梯度
    """
    caches_list, x = caches
    n_a, m, T_x = da.shape
    n_x = x.shape[0]

    ### START CODE HERE ###



    ### END CODE HERE ###

    gradients = {
        'dx': dx, 'da0': da0, 'dc0': dc0,
        'dWf': dWf, 'dbf': dbf, 'dWi': dWi, 'dbi': dbi,
        'dWc': dWc, 'dbc': dbc, 'dWo': dWo, 'dbo': dbo
    }
    return gradients
